# Compute Embeddings — Detecção de Drift (Colab)

Adaptador fino que monta o Google Drive, atualiza o repositório e chama `scripts/drift/compute_embeddings.py` (ADR 011 §D.1 + §D.6). **Nenhuma lógica científica vive neste notebook** (CLAUDE.md §5).

Gera o cache único de embeddings BERTimbau `[CLS]` consumido pelos blocos B1 (estatístico), B2 (semântico) e B3 (CPD) da pipeline de drift.

**Pré-requisitos:**
- `data/processado/corpus_opcao7.parquet` já gerado por `preprocessar.ipynb` (no Drive em `MyDrive/ptbr-market-classification/data/processado/`).
- Runtime → Change runtime type → **GPU (T4 ou L4)**. Em CPU o script aborta; use `--permitir-cpu` apenas para smoke test.

**Saída** em `MyDrive/ptbr-market-classification/artifacts/drift/embeddings/bertimbau_base_cls/`:
- `metadata.json`: configuração, hashes, versões, duração.
- `embeddings.parquet`: `link`, `date`, `embedding` (list[float32] × 768).

**Tempo esperado** (corpus ~160k artigos, max_len=256, batch=64):
- L4: ~10–15 min.
- T4: ~25–40 min.
- CPU: inviável (~horas).

## 1. Parâmetros (editar conforme necessário)

In [ ]:
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification-2.git'  # substituir pela URL do seu fork
RAMO = 'main'

DIR_REPO = '/content/ptbr-market-classification'
DIR_DRIVE = '/content/drive/MyDrive/ptbr-market-classification'

CAMINHO_CORPUS = f'{DIR_DRIVE}/data/processado/corpus_opcao7.parquet'
DIR_ARTEFATOS_DRIFT = f'{DIR_DRIVE}/artifacts/drift'

# Para smoke test rápido, defina um valor (ex: 200) — útil para validar o
# pipeline antes do run completo. None = todos os artigos.
LIMITE_ARTIGOS = None

# Batch de inferência. 64 é seguro em T4/L4; subir para 128 se VRAM permitir.
BATCH_SIZE = 64

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar / atualizar repositório

In [ ]:
import os, subprocess

if os.path.exists(DIR_REPO):
    subprocess.run(['git', '-C', DIR_REPO, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'checkout', RAMO], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', RAMO, REPO_URL, DIR_REPO], check=True)

os.chdir(DIR_REPO)
print('cwd =', os.getcwd())

## 4. Instalar dependências

In [ ]:
!pip install -q -r requirements.txt

## 5. Validar GPU

Aborta cedo se não houver GPU — evita gastar horas em CPU sem aviso.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'GPU não detectada. Vá em Runtime → Change runtime type e escolha T4 ou L4.'
)
nome_gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {nome_gpu} ({vram_gb:.1f} GB VRAM)')

## 6. Verificar presença do corpus parquet

In [ ]:
from pathlib import Path

caminho = Path(CAMINHO_CORPUS)
assert caminho.exists(), (
    f'Corpus parquet não encontrado em {caminho}. '
    'Rode preprocessar.ipynb antes (ou ajuste CAMINHO_CORPUS).'
)
print(f'Corpus: {caminho} ({caminho.stat().st_size / 1024**2:.1f} MB)')

## 7. Executar `compute_embeddings.py`

Variáveis de ambiente sobrescrevem os defaults de `src.config`, mantendo todos os artefatos no Drive.

In [ ]:
import os

os.environ['PTBR_MC_DIR_DRIFT'] = DIR_ARTEFATOS_DRIFT

args = [
    '--corpus', CAMINHO_CORPUS,
    '--out', DIR_ARTEFATOS_DRIFT,
    '--batch-size', str(BATCH_SIZE),
]
if LIMITE_ARTIGOS is not None:
    args += ['--limite-artigos', str(LIMITE_ARTIGOS)]

cmd = 'python scripts/drift/compute_embeddings.py ' + ' '.join(args)
print('Executando:', cmd)
!{cmd}

## 8. Resumo do artefato gerado

In [ ]:
import json
from pathlib import Path
import pandas as pd

dir_emb = Path(DIR_ARTEFATOS_DRIFT) / 'embeddings' / 'bertimbau_base_cls'
print('Diretório:', dir_emb)
for p in sorted(dir_emb.glob('*')):
    print(f'  {p.name} ({p.stat().st_size / 1024**2:.2f} MB)')

print('\nMetadata (campos principais):')
metadata = json.loads((dir_emb / 'metadata.json').read_text())
for chave in ('bloco', 'escopo', 'seed_global', 'git_commit',
              'timestamp_iso', 'duracao_segundos', 'corpus', 'extras'):
    print(f'  {chave}: {metadata.get(chave)}')

print('\nSmoke check do parquet:')
df = pd.read_parquet(dir_emb / 'embeddings.parquet', engine='pyarrow')
print(f'  linhas: {len(df):,}')
print(f'  colunas: {list(df.columns)}')
print(f'  shape do primeiro embedding: {len(df.iloc[0]["embedding"])}')
print(f'  exemplo de link/date: {df.iloc[0]["link"]} / {df.iloc[0]["date"]}')